<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%205/5.2%20Agents%20with%20PydanticAI/5.2.2%20Tutorial%20-%20Reactive%20Agent%20Flow%20with%20PydanticAI%20(Step-by-Step)_OpenRouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "pydantic-ai-slim[openrouter]==2.48.0" openai==3.19.0 pydantic==2.13.5 python-dotenv==1.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 668.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.6/146.6 kB 5.4 MB/s eta 0:00:00


## Reactive Agent Flow with PydanticAI (Step-by-Step)

In this tutorial, we'll build an agent that can think, plan, and adapt its approach
based on what it discovers. This is where agents become truly intelligent.

Think of this like teaching an agent to be a detective - it starts with a question,
gathers clues, analyzes what it finds, and then decides what to investigate next.

Learning Objectives:
- Understand reactive vs. scripted agent behavior
- Implement multi-step reasoning workflows
- Build agents that adapt based on intermediate results
- Create decision trees for complex problem solving

In [2]:
import os
from typing import List, Optional, Dict, Any, Union
from datetime import datetime
from enum import Enum
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
import asyncio
import json

# Load environment variables
load_dotenv()

False

In [4]:
# OpenRouter setup
from getpass import getpass
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
MODEL = "openai/gpt-4.1-mini"

# PydanticAI model routed through OpenRouter; used by every agent below.
openrouter_model = OpenRouterModel(MODEL, provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY))

Enter your OpenRouter API key: ··········


### Understanding Reactive vs. Scripted Agents

Let's start by understanding the difference between agents that follow a script
versus agents that react and adapt to what they discover.


In [5]:
class TaskType(str, Enum):
    """
    Different types of tasks our reactive agent can handle.
    Each type requires a different approach and reasoning pattern.
    """
    RESEARCH = "research"
    ANALYSIS = "analysis"
    PROBLEM_SOLVING = "problem_solving"
    PLANNING = "planning"
    DECISION_MAKING = "decision_making"

class StepResult(BaseModel):
    """
    Represents the outcome of a single step in our agent's reasoning process.
    """
    step_number: int = Field(description="Which step this is in the process")
    action_taken: str = Field(description="What action the agent performed")
    findings: str = Field(description="What the agent discovered")
    confidence: float = Field(description="How confident the agent is in this step", ge=0.0, le=1.0)
    next_action: Optional[str] = Field(description="What the agent plans to do next")
    should_continue: bool = Field(description="Whether the agent should continue the process")

class ReasoningTrace(BaseModel):
    """
    Complete trace of the agent's reasoning process from start to finish.
    """
    task_type: TaskType
    initial_query: str
    steps: List[StepResult] = Field(description="All steps the agent took")
    final_conclusion: str = Field(description="The agent's final answer or recommendation")
    total_confidence: float = Field(description="Overall confidence in the final result", ge=0.0, le=1.0)
    reasoning_path: List[str] = Field(description="High-level description of the reasoning path taken")


### Creating the Reactive Agent Foundation

Now let's create an agent that can reason step-by-step and adapt its approach.


In [6]:
reactive_agent = Agent(
    openrouter_model,
    output_type=ReasoningTrace,
    system_prompt="""
    You are a reactive reasoning agent that approaches problems methodically.

    Your process:
    1. Understand the task and determine what type of reasoning is needed
    2. Break down complex problems into manageable steps
    3. Gather information or perform analysis for each step
    4. Evaluate what you've learned and decide the next action
    5. Adapt your approach based on what you discover
    6. Continue until you reach a confident conclusion

    Key principles:
    - Be explicit about your reasoning at each step
    - Don't jump to conclusions - build evidence gradually
    - If you hit a dead end, backtrack and try a different approach
    - Always assess your confidence and communicate uncertainty
    - Stop when you have sufficient information to provide a good answer
    """
)

### Information Gathering Tools

Our reactive agent needs tools to gather information at each step.


In [7]:

@reactive_agent.tool
async def gather_background_info(ctx: RunContext[None], topic: str, focus_area: str) -> str:
    """
    Gather background information about a topic, focusing on a specific area.
    """
    # Simulated knowledge base for demonstration
    info_database = {
        "electric_vehicles": {
            "technology": "EVs use battery packs to store electricity and electric motors for propulsion. Key components include lithium-ion batteries, regenerative braking, and charging systems.",
            "market": "EV market growing rapidly, with Tesla leading but traditional automakers catching up. Government incentives driving adoption.",
            "environment": "EVs produce zero direct emissions but environmental impact depends on electricity source and battery production.",
            "economics": "Higher upfront costs but lower operating costs. Battery costs falling rapidly, approaching price parity with ICE vehicles."
        },
        "artificial_intelligence": {
            "technology": "AI encompasses machine learning, neural networks, natural language processing, and computer vision. Current focus on large language models and transformer architectures.",
            "applications": "AI used in healthcare, finance, transportation, education, and entertainment. Growing automation of cognitive tasks.",
            "ethics": "Concerns about bias, privacy, job displacement, and AI alignment. Need for responsible AI development and regulation.",
            "future": "Potential for AGI, but timeline uncertain. Current focus on improving capabilities while addressing safety concerns."
        },
        "climate_change": {
            "science": "Global warming caused by greenhouse gas emissions, primarily CO2 from fossil fuels. Temperature rise of 1.1°C since pre-industrial times.",
            "impacts": "Rising sea levels, extreme weather, ecosystem disruption, agricultural impacts, human displacement.",
            "solutions": "Renewable energy transition, carbon capture, energy efficiency, electrification, policy changes.",
            "economics": "High costs of inaction outweigh costs of action. Clean energy increasingly cost-competitive."
        }
    }

    topic_key = topic.lower().replace(" ", "_")
    topic_info = info_database.get(topic_key, {})

    if focus_area.lower() in topic_info:
        return f"Background info on {topic} ({focus_area}): {topic_info[focus_area.lower()]}"
    else:
        return f"Limited background information available for {focus_area} aspect of {topic}"


### Analysis and Reasoning Tools

Let's add tools that help our agent analyze and reason about information.


In [8]:

@reactive_agent.tool
async def analyze_pros_and_cons(ctx: RunContext[None], topic: str, context: str) -> str:
    """
    Analyze the advantages and disadvantages of something based on context.
    """
    # This would normally use more sophisticated analysis
    analysis_prompts = {
        "pros": f"Based on the context about {topic}, what are the main advantages or positive aspects?",
        "cons": f"Based on the context about {topic}, what are the main disadvantages or concerns?"
    }

    # Simulated analysis - in practice, this might call another AI model or use structured analysis
    example_analysis = f"""
    Pros and Cons Analysis for {topic}:

    PROS:
    - Potential benefits identified from context analysis
    - Positive trends and opportunities
    - Stakeholder advantages

    CONS:
    - Challenges and limitations identified
    - Potential risks and downsides
    - Implementation barriers

    Context considered: {context[:100]}...
    """

    return example_analysis

In [9]:

@reactive_agent.tool
async def evaluate_evidence_strength(ctx: RunContext[None], evidence: str, claim: str) -> str:
    """
    Evaluate how well evidence supports a particular claim.
    """
    # Simple evidence evaluation logic
    evidence_indicators = {
        "strong": ["research shows", "studies indicate", "data demonstrates", "peer-reviewed"],
        "moderate": ["experts suggest", "analysis indicates", "trends show", "reports"],
        "weak": ["some believe", "it's thought", "anecdotal", "rumors"]
    }

    evidence_lower = evidence.lower()
    strength = "unknown"

    for level, indicators in evidence_indicators.items():
        if any(indicator in evidence_lower for indicator in indicators):
            strength = level
            break

    return f"""
    Evidence Evaluation:
    Claim: {claim}
    Evidence Strength: {strength.upper()}

    Evidence excerpt: {evidence[:200]}...

    Assessment: The evidence appears to be {strength} based on language and source indicators.
    """

### Decision Making and Path Planning

Our agent needs to make decisions about what to do next based on what it learns.


In [10]:

@reactive_agent.tool
async def plan_next_steps(ctx: RunContext[None], current_findings: str, remaining_questions: List[str]) -> str:
    """
    Plan what the agent should investigate next based on current findings.
    """
    if not remaining_questions:
        return "No remaining questions - ready to conclude analysis"

    # Prioritize questions based on importance and feasibility
    priority_assessment = []
    for i, question in enumerate(remaining_questions):
        # Simple priority scoring
        priority_score = len(question.split())  # Longer questions might be more complex
        if any(word in question.lower() for word in ["critical", "important", "key"]):
            priority_score += 5
        if any(word in question.lower() for word in ["how", "why", "what causes"]):
            priority_score += 3

        priority_assessment.append({
            "question": question,
            "priority": priority_score,
            "rationale": f"Score: {priority_score} based on complexity and keywords"
        })

    # Sort by priority
    priority_assessment.sort(key=lambda x: x["priority"], reverse=True)

    next_steps = f"""
    Next Steps Planning:

    Current findings summary: {current_findings[:150]}...

    Remaining questions prioritized:
    """

    for i, item in enumerate(priority_assessment[:3], 1):  # Show top 3
        next_steps += f"\n{i}. {item['question']} ({item['rationale']})"

    next_steps += f"\n\nRecommended next action: Investigate highest priority question first"

    return next_steps

### Creating a Complete Reactive Workflow

Let's put everything together to create a truly reactive agent workflow.


In [11]:
# A complex query that requires multi-step reasoning
complex_query = """
I'm considering whether my city should invest in electric vehicle charging infrastructure.
I need to understand the technology, costs, benefits, and potential challenges to make
a recommendation to the city council.
"""

print(f"Complex Query: {complex_query}\n")
print("🤖 Agent starting reactive reasoning process...\n")

# Run the reactive agent
result = await reactive_agent.run(complex_query)

# Display the reasoning trace
print("="*70)
print("REACTIVE REASONING TRACE")
print("="*70)

print(f"Task Type: {result.output.task_type.value}")
print(f"Initial Query: {result.output.initial_query}")
print(f"Final Confidence: {result.output.total_confidence:.1%}\n")

print("🛤️ REASONING PATH:")
for i, path_step in enumerate(result.output.reasoning_path, 1):
    print(f"{i}. {path_step}")
print()

print("📝 DETAILED STEPS:")
for step in result.output.steps:
    print(f"\nStep {step.step_number}: {step.action_taken}")
    print(f"Findings: {step.findings}")
    print(f"Confidence: {step.confidence:.1%}")
    if step.next_action:
        print(f"Next Action: {step.next_action}")
    print(f"Continue: {'Yes' if step.should_continue else 'No'}")

print(f"\n🎯 FINAL CONCLUSION:")
print(result.output.final_conclusion)

Complex Query: 
I'm considering whether my city should invest in electric vehicle charging infrastructure. 
I need to understand the technology, costs, benefits, and potential challenges to make 
a recommendation to the city council.


🤖 Agent starting reactive reasoning process...

REACTIVE REASONING TRACE
Task Type: research
Initial Query: city investment in electric vehicle charging infrastructure, including technology, costs, benefits, and challenges
Final Confidence: 50.0%

🛤️ REASONING PATH:
1. Attempt to gather background information on key aspects failed due to limited data.
2. General pros and cons were analyzed to frame the issue.
3. Next steps planned to focus on prioritized specific questions.
4. Repeated attempts to gather specific background information also lacked depth.
5. Conclusion based on general understanding and recommendation for further targeted local investigation.

📝 DETAILED STEPS:

Step 1: Gather background information on technology and costs of electric veh